### Block 0 — Install dependencies

In [1]:
!pip install -q datasets huggingface_hub scikit-learn tqdm pandas requests


### Block 1 — Load FPB dataset ChanceFocus/en-fpb

In [2]:
import os
import getpass

from datasets import load_dataset
from huggingface_hub import HfApi, HfFolder

DATASET_NAME = "ChanceFocus/en-fpb"

# Get HF token: env -> Colab userdata -> manual input
hf_token = HfFolder.get_token() or os.getenv("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (hf_...): ")

# Optional: verify access
api = HfApi()
info = api.dataset_info(DATASET_NAME, token=hf_token)
HfFolder.save_token(hf_token)
print(f"Access to {DATASET_NAME} OK ✅")
print("Files/siblings:", [s.rfilename for s in info.siblings])

# Load all splits
ds_all = load_dataset(DATASET_NAME, token=hf_token)

print("\nLoaded splits:")
for split_name, split_ds in ds_all.items():
    print(f"  {split_name}: {len(split_ds)} examples")

# Prefer 'test' if it exists, otherwise first split
SPLIT = "test" if "test" in ds_all else list(ds_all.keys())[0]
dataset = ds_all[SPLIT]

print(f"\nUsing split: {SPLIT}")
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Hugging Face token (hf_...): ··········
Access to ChanceFocus/en-fpb OK ✅
Files/siblings: ['.gitattributes', 'README.md', 'data/test-00000-of-00001-8bd1e21c671fb670.parquet', 'data/train-00000-of-00001-ab9a3b4799b09589.parquet', 'data/valid-00000-of-00001-303e4ba2afe838d4.parquet']


README.md:   0%|          | 0.00/742 [00:00<?, ?B/s]

data/train-00000-of-00001-ab9a3b4799b095(…):   0%|          | 0.00/608k [00:00<?, ?B/s]

data/test-00000-of-00001-8bd1e21c671fb67(…):   0%|          | 0.00/188k [00:00<?, ?B/s]

data/valid-00000-of-00001-303e4ba2afe838(…):   0%|          | 0.00/154k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/776 [00:00<?, ? examples/s]


Loaded splits:
  train: 3100 examples
  test: 970 examples
  valid: 776 examples

Using split: test
Number of examples: 970
Example 0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


### Block 2 — Label config + prediction normalization

In [3]:
import re
from typing import Optional

from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import pandas as pd

LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: Optional[str]) -> str:
    """
    Map a free-form model output to one of the three labels.
    """
    if raw is None:
        return "neutral"
    text = raw.strip().lower()

    # Exact match
    for label in LABELS:
        if text == label:
            return label

    # Whole-word match
    for label in LABELS:
        if re.search(r"\b" + re.escape(label) + r"\b", text):
            return label

    # Fallback
    return "neutral"


### Block 3 — Qwen2.5-32B Inference client + single-sentence classifier

In [6]:
import time
from huggingface_hub import InferenceClient

MODEL_ID_QWEN = "Qwen/Qwen2.5-32B-Instruct"

# Reuse the same HF token we used for the dataset
client_qwen = InferenceClient(model=MODEL_ID_QWEN, token=hf_token)

SYSTEM_INSTRUCTION = (
    "You are a financial sentiment classifier.\n"
    "Given a single sentence from a financial news article or microblog,\n"
    "classify its overall sentiment from the perspective of an investor as\n"
    "exactly one of: negative, neutral, or positive.\n"
    "Respond with only one word: negative, neutral, or positive."
)

def classify_with_qwen(
    sentence: str,
    max_retries: int = 6,
    sleep_base: float = 1.0,
) -> str:
    """
    Call Qwen2.5-32B-Instruct via Hugging Face Inference API
    using the conversational (chat) interface, and return a
    normalized three-way sentiment label.
    """
    last_err = None

    for attempt in range(max_retries):
        try:
            messages = [
                {"role": "system", "content": SYSTEM_INSTRUCTION},
                {"role": "user", "content": sentence},
            ]

            resp = client_qwen.chat_completion(
                messages=messages,
                max_tokens=16,
                temperature=0.0,
                stream=False,
            )

            text_out = resp.choices[0].message["content"]
            return normalize_prediction(text_out)
        except Exception as e:
            last_err = e
            wait = sleep_base * (2 ** attempt)
            print(f"[Qwen2.5-32B] Error on attempt {attempt+1}/{max_retries}: {last_err}")
            time.sleep(wait)

    print("[Qwen2.5-32B] Failed after retries, returning 'neutral'.")
    return "neutral"


### Block 4 — Evaluation loop on FPB

In [8]:
y_true = []
y_pred = []

print(f"Evaluating model: {MODEL_ID_QWEN} on {len(dataset)} FPB examples...\n")

for example in tqdm(dataset, total=len(dataset)):
    sentence = example["text"]                         # change if your text column has another name
    true_label = example["answer"].lower().strip()     # change if your label column has another name
    pred_label = classify_with_qwen(sentence)

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total examples:", len(dataset))
print("Got predictions for:", len(y_pred))


Evaluating model: Qwen/Qwen2.5-32B-Instruct on 970 FPB examples...



100%|██████████| 970/970 [23:21<00:00,  1.44s/it]

Total examples: 970
Got predictions for: 970


### Block 5 — Metrics + sample errors

In [9]:
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))

df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))



Overall accuracy: 0.8237

Classification report:
              precision    recall  f1-score   support

    negative       0.80      0.91      0.85       116
     neutral       0.87      0.83      0.85       577
    positive       0.75      0.77      0.76       277

    accuracy                           0.82       970
   macro avg       0.80      0.84      0.82       970
weighted avg       0.83      0.82      0.82       970


First 10 predictions:
                                                                                                                                                                                                                                                                                 text     gold     pred
                                                                                           The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functio